# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Torch is usually pre-installed on Colab, just ensuring torchvision
    !pip install -q pillow
    !pip install -q torchvision --index-url https://download.pytorch.org/whl/cpu
    print('Packages installed.')


# Image Analysis In-Depth
This notebook demonstrates how to load, process, and classify images using PyTorch and Torchvision.

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load an Image
We download a sample image from the web (a cat).

In [ ]:
url = "https://upload.wikimedia.org/wikipedia/commons/4/4d/Cat_November_2010-1a.jpg"
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
img = Image.open(BytesIO(response.content))

plt.imshow(img)
plt.axis('off')
plt.show()

## 2. Preprocess for ResNet
Models expect images to be normalized in a specific way.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

img_t = preprocess(img)
batch_t = torch.unsqueeze(img_t, 0).to(device)

## 3. Classification
We use a pre-trained ResNet18 model to classify the image.

In [ ]:
# Load model
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.eval()
resnet.to(device)

# Predict
out = resnet(batch_t)

# Load labels
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.split('\n')

# Get top 3
percentage = torch.nn.functional.softmax(out, dim=1)[0] * 100
_, indices = torch.sort(out, descending=True)

print("Top 3 Predictions:")
for idx in indices[0][:3]:
    print(f"{labels[idx]}: {percentage[idx].item():.2f}%")